# Models: baselines and the course model

This notebook fits each model on the training split and scores it on the validation split. Every model uses the same interface and the same two metrics, so the rows of the results table are comparable.

The trivial baseline gives every row the class frequencies of the training set, so it always answers "fit". It sets the floor: a model that cannot beat it has learned nothing.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

import pandas as pd  # noqa: E402

from src import data, metrics, models  # noqa: E402

## Data and split

`src.data.load()` reads the raw file, or the small sample when `DATA_DIR` points at `tests/fixtures`. `src.data.split()` returns the shared 80/10/10 split with seed 158. This notebook leaves the test split alone; the final evaluation scores it once.

In [ ]:
df = data.load()
train, valid, test = data.split(df)
print(f"train {len(train)}, valid {len(valid)}")
train["fit"].value_counts(normalize=True).round(3)

In [ ]:
train_p, valid_p, _ = data.preprocess(
    train, valid, test
)  # fills use train statistics only; test is not scored here

## Fit and score

Every model returns a probability for each class, because the primary metric (average one-vs-rest AUC) ranks scores. Macro-F1 uses the hard labels, which are the most likely class. The majority baseline gives the same probabilities to every row, so its AUC is 0.5 by construction. Macro-F1 shows what always saying "fit" costs: the two minority classes score 0.

In [ ]:
baseline = models.MajorityClass().fit(train_p)
baseline_row = {
    "average AUC": metrics.average_auc(valid_p["fit"], baseline.predict_proba(valid_p)),
    "macro-F1": metrics.macro_f1(valid_p["fit"], baseline.predict(valid_p)),
}
baseline_row

## Course model: multinomial logistic regression

**Inputs.** One row per transaction after `data.preprocess()`: the garment `size`, the user's `weight_lbs`, `height_in`, `bust_band`, `cup` and `age` with a 0/1 missing flag for each, and the one-hot columns `body_type`, `category` and `rented_for`. Numbers are standardised. The rating and the review text are not used, because they are written after the fit is known.

**Output.** A probability for each of small, fit and large, taken from a softmax over three linear scores $z_k = w_k^\top x + b_k$:

$$p(y = k \mid x) = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

**Objective.** Fit $W$ and $b$ on the training split by minimising the L2-regularised cross-entropy:

$$\min_{W,b}\; \frac{1}{2C}\lVert W \rVert_2^2 + \sum_{i=1}^{n} -\log p(y_i \mid x_i)$$

$C = 1$ and equal class weights are the defaults. They are not tuned here. The model sees a user's measurements and the size, but no user or item identity, so it cannot learn that a particular dress runs small.

In [ ]:
logreg = models.LogisticRegression().fit(train_p)
logreg_row = {
    "average AUC": metrics.average_auc(valid_p["fit"], logreg.predict_proba(valid_p)),
    "macro-F1": metrics.macro_f1(valid_p["fit"], logreg.predict(valid_p)),
}
results = pd.DataFrame([baseline_row, logreg_row], index=["MajorityClass", "LogisticRegression"]).round(4)
print(results)
results

## Reading the table

Compare the two rows on both metrics. The majority baseline scores an AUC of exactly 0.5, so any AUC above 0.5 for logistic regression is signal from size and measurements. Macro-F1 can still stay low if the model rarely predicts the rare classes: with about 74% of labels being "fit", the most likely class is usually "fit". The numbers in this table come from whichever file `DATA_DIR` points at, so quote them only after a run on the full data.

## Adding a model

A new model subclasses `models.Model`, implements `fit(train)` and `predict_proba(df)`, and gets `predict` for free. Add a row to the results table and nothing else changes.

## Main model: user and item latent factors

**Why identities.** The course model sees measurements and size but not which dress was rented, so it cannot learn that one dress runs small. Rent the Runway repeats items many times, so an item identity can carry that signal. `LatentFactorFit` keeps the same feature matrix as the course model and adds a score per user and per item.

**Score.** For class $k \in \{\text{small}, \text{fit}, \text{large}\}$, user $u$ and item $i$:

$$z_k = w_k^\top x + b_k + b^{user}_{u,k} + b^{item}_{i,k} + \gamma_u^\top \delta_{i,k}$$

$x$ is the feature vector from the course model. $b^{user}$ and $b^{item}$ are one bias per class. $\gamma_u \in \mathbb{R}^K$ is a user vector shared across classes, and $\delta_{i,k} \in \mathbb{R}^K$ is an item vector for each class. The probabilities are the softmax of $z$. With $K = 0$ the last term disappears and the model is logistic regression plus user and item biases.

**Objective.** Class weights $s_i$ come from `class_weight="balanced"`, which makes each class count equally in the loss:

$$\min \; \sum_i s_i \,(-\log p(y_i \mid x_i, u_i, i_i)) + \tfrac{1}{2}\lVert W \rVert^2 + \tfrac{\lambda}{2}\left(\lVert b^{user} \rVert^2 + \lVert b^{item} \rVert^2 + \lVert \gamma \rVert^2 + \lVert \delta \rVert^2\right)$$

The penalty on the user and item terms pulls an identity with few transactions toward zero. A user or item that never appeared in train gets zero bias and zero factors, so the model falls back to the features alone.

**Fitting.** L-BFGS on the full training set with exact gradients (`scipy.optimize.minimize`). The loss is smooth, there is no learning rate to tune, and the sparse indicator matrices keep each step fast. Minibatch SGD would need a learning rate schedule and gives noisier results for little gain at this size.

In [ ]:
import time

for name, part in (("users", "user_id"), ("items", "item_id")):
    seen = valid_p[part].isin(train_p[part]).mean()
    print(f"share of valid rows whose {name[:-1]} appears in train: {seen:.3f}")
print("transactions per user in train, median:", train_p["user_id"].value_counts().median())
print("transactions per item in train, median:", train_p["item_id"].value_counts().median())

## Three approaches compared

| Approach | Complexity | Efficiency | Implementation difficulty |
|---|---|---|---|
| Logistic regression with `class_weight="balanced"` | Features only: a few dozen weights per class | Fastest: one scikit-learn call | Lowest: the course model with one argument |
| Latent factor model with $K = 0$ | Features plus one bias per user and item and class: parameters grow with the number of identities | One L-BFGS run over sparse indicator matrices, slower than the first row | Medium: custom loss and gradient |
| Latent factor model with $K > 0$ | Adds a $K$-dim vector per user and a $3K$ block per item | Slowest: more parameters and a dense product per step | Highest: the gradient of the interaction term, checked against finite differences |

The training seconds in the table below are measured with `time.perf_counter` on this machine, on the file that `DATA_DIR` points at.

In [ ]:
approaches = {
    "MajorityClass": models.MajorityClass(),
    "LogisticRegression": models.LogisticRegression(),
    "LogisticRegression balanced": models.LogisticRegression(class_weight="balanced"),
    "LatentFactorFit K=0 balanced": models.LatentFactorFit(K=0, class_weight="balanced"),
    "LatentFactorFit K=4 balanced": models.LatentFactorFit(K=4, class_weight="balanced"),
}
rows = {}
for name, model in approaches.items():
    start = time.perf_counter()
    model.fit(train_p)
    seconds = time.perf_counter() - start
    if isinstance(model, models.LatentFactorFit):
        print(f"{name}: {model.n_iter_} iterations, converged {model.converged_}")
    rows[name] = {
        "average AUC": metrics.average_auc(valid_p["fit"], model.predict_proba(valid_p)),
        "macro-F1": metrics.macro_f1(valid_p["fit"], model.predict(valid_p)),
        "train seconds": seconds,
    }
results_main = pd.DataFrame(rows).T.round(4)
print(results_main)
results_main

In [ ]:
# Which identity matters? Blank one id column in train and valid (every row gets the same id).
ablation = {}
for name, blank in (("item biases only", "user_id"), ("user biases only", "item_id")):
    tr, va = train_p.assign(**{blank: "x"}), valid_p.assign(**{blank: "x"})
    model = models.LatentFactorFit(K=0, class_weight="balanced").fit(tr)
    print(f"{name}: {model.n_iter_} iterations, converged {model.converged_}")
    ablation[name] = {
        "average AUC": metrics.average_auc(va["fit"], model.predict_proba(va)),
        "macro-F1": metrics.macro_f1(va["fit"], model.predict(va)),
    }
pd.DataFrame(ablation).T.round(4)

## Reading the main-model table

Validation only: the test split is not scored here. The committed copy has its outputs cleared and runs on the small sample in CI. The findings below come from running it on the full data with `DATA_DIR` set to `data/raw`.

- **Class weighting** moves macro-F1 more than AUC. The model predicts small and large more often, which the unweighted argmax almost never does.
- **Identities are the big gain.** Both LatentFactorFit rows score a clearly higher AUC than both LogisticRegression rows. Adding user and item biases helps far more than any measurement feature did.
- **The item carries it.** In the ablation, item biases alone come close to the full model and user biases alone stay well below it. Most users rent once, so a user bias is fitted to a single transaction and the penalty shrinks it toward zero. Items recur, so their biases are estimated from several outcomes.
- **Latent vectors did not help.** The $K = 4$ row does not beat the $K = 0$ row and takes longer to train. A vector per user cannot be learned from one transaction, and the item bias already holds most of what an item says about fit. I did not force a result: the simpler $K = 0$ model is the better one here.
- **Convergence.** Each LatentFactorFit fit prints its iteration count and whether L-BFGS converged, and warns when it stops at the iteration cap. A fit that stops at the cap is a partly trained model, so read those lines before trusting a row. I-11 sets the cap.

$\lambda$, $K$ and `class_weight` are set by hand. I-11 tunes them on validation.

## Tuning on the validation split

Each model has a few settings that the training loss cannot choose: the penalty strength, the class weights and, for the latent factor model, the number of factors $K$. This section fits each setting on train and scores it on valid. The test split is not touched.

**Rule.** The setting with the highest validation average AUC wins, because AUC is the primary metric in `docs/decision.md`. Macro-F1 is shown next to it and breaks no ties, but a chosen setting that loses badly on macro-F1 would deserve a second look.

**Grids.**

- `LogisticRegression`: `C` in {0.01, 0.1, 1, 10} against `class_weight` in {none, balanced}. 8 fits.
- `LatentFactorFit`, stage 1: $\lambda$ in {1, 2, 5, 10, 20} against `class_weight` in {none, balanced}, with $K = 0$. 10 fits.
- `LatentFactorFit`, stage 2: $K$ in {2, 4, 8}, with $\lambda$ and `class_weight` fixed at the stage 1 winner. 3 fits, and the $K = 0$ point comes from stage 1.

**What I cut.** The two stages search one setting at a time, not the full $\lambda \times K \times$ class weight product (30 fits, about twice the runtime). The best $\lambda$ for $K > 0$ could differ from the best for $K = 0$. L-BFGS stays at 200 iterations, and no feature subsets are tried.

In [ ]:
GRID_C = [0.01, 0.1, 1, 10]
GRID_LAMBDA = [1, 2, 5, 10, 20]
GRID_K = [2, 4, 8]
CLASS_WEIGHTS = {"none": None, "balanced": "balanced"}


def score(model):
    start = time.perf_counter()
    model.fit(train_p)
    seconds = time.perf_counter() - start
    return {
        "average AUC": metrics.average_auc(valid_p["fit"], model.predict_proba(valid_p)),
        "macro-F1": metrics.macro_f1(valid_p["fit"], model.predict(valid_p)),
        "train seconds": seconds,
    }


lr_grid = pd.DataFrame(
    [
        {"C": C, "class_weight": name, **score(models.LogisticRegression(C=C, class_weight=cw))}
        for C in GRID_C
        for name, cw in CLASS_WEIGHTS.items()
    ]
)
lr_grid.round(4)

In [ ]:
lff_lam = pd.DataFrame(
    [
        {"lam": lam, "class_weight": name, **score(models.LatentFactorFit(K=0, lam=lam, class_weight=cw))}
        for lam in GRID_LAMBDA
        for name, cw in CLASS_WEIGHTS.items()
    ]
)
stage1 = lff_lam.loc[lff_lam["average AUC"].idxmax()]
lam1, cw1 = stage1["lam"], stage1["class_weight"]
lff_k = pd.DataFrame(
    [{"K": 0, "lam": lam1, "class_weight": cw1, **stage1[["average AUC", "macro-F1", "train seconds"]]}]
    + [
        {
            "K": K,
            "lam": lam1,
            "class_weight": cw1,
            **score(models.LatentFactorFit(K=K, lam=lam1, class_weight=CLASS_WEIGHTS[cw1])),
        }
        for K in GRID_K
    ]
)
print(lff_lam.round(4).to_string(index=False))
print()
print(lff_k.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

best_lr = lr_grid.loc[lr_grid["average AUC"].idxmax()]
best_lff = lff_k.loc[lff_k["average AUC"].idxmax()]
BEST = {
    "LogisticRegression": {"C": float(best_lr["C"]), "class_weight": CLASS_WEIGHTS[best_lr["class_weight"]]},
    "LatentFactorFit": {
        "K": int(best_lff["K"]),
        "lam": float(best_lff["lam"]),
        "class_weight": CLASS_WEIGHTS[best_lff["class_weight"]],
    },
}

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex="col")
panels = [
    ("C", lr_grid, "class_weight", best_lr, "LogisticRegression: C", True),
    ("lam", lff_lam, "class_weight", stage1, "LatentFactorFit (K=0): lambda", True),
    ("K", lff_k, None, best_lff, "LatentFactorFit: K", False),
]
for col, (x, grid, by, chosen, title, logx) in enumerate(panels):
    for row, metric in enumerate(["average AUC", "macro-F1"]):
        ax = axes[row, col]
        for label, part in grid.groupby(by) if by else [("", grid)]:
            ax.plot(part[x], part[metric], marker="o", label=label or None)
        ax.scatter([chosen[x]], [chosen[metric]], s=200, facecolors="none", edgecolors="black", zorder=3)
        if logx:
            ax.set_xscale("log")
        ax.set_ylabel(metric)
        if row == 0:
            ax.set_title(title)
        if row == 1:
            ax.set_xlabel(x)
        if by and col == 0 and row == 0:
            ax.legend(title="class_weight")
fig.suptitle("Validation score vs setting (circled: highest average AUC in its grid)")
fig.tight_layout()
plt.show()

print("chosen:", BEST)
print("matches models.BEST_PARAMS:", BEST == models.BEST_PARAMS)

## Chosen settings

The settings are stored in `models.BEST_PARAMS`, keyed by class name, so the final evaluation builds each model with `getattr(models, name)(**params)`. The last cell above prints whether a rerun still agrees with the stored values. On the small CI sample it will not, because the validation split there has a few dozen rows. The stored values come from the full data.

**Findings on the full data.**

- **Logistic regression barely moves.** Across all 8 settings, AUC stays between 0.6171 and 0.6188. `C = 0.01` with balanced weights is the top row, but its lead over the default is 0.0002, too small to matter. Balanced weights lift macro-F1 from about 0.29 to about 0.33 and leave AUC unchanged.
- **The latent factor model prefers a light penalty.** At $K = 0$, AUC peaks at $\lambda = 2$ (0.7570) and falls to 0.7405 at $\lambda = 20$. Balanced weights beat no weights at every $\lambda$, on AUC as well as macro-F1.
- **More factors lower AUC.** With $\lambda = 2$ and balanced weights, AUC goes 0.7570 ($K = 0$), 0.7484, 0.7439, 0.7368 ($K = 8$) while training time grows from 14 to 40 seconds. Macro-F1 rises slightly, from 0.518 to 0.531. The search varied one setting at a time, so a larger $\lambda$ paired with $K > 0$ was not tried.

The chosen main model is therefore the bias-only model: $K = 0$, $\lambda = 2$, balanced class weights. The test split is not used anywhere in this section.